In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt

import os
from pathlib import Path
import datetime
import warnings

from vivarium_helpers.utils import convert_to_categorical, constant_categorical, print_memory_usage, Timer

from vivarium_helpers.projects.alzheimers import loading, population
from vivarium_helpers.projects.alzheimers.population import RunType
from vivarium_helpers.projects.alzheimers.results import AlzheimersResultsProcessor

!date
!whoami
!pwd

Thu Nov 19 17:31:39 PST 2026


lutzes


/ihme/homes/lutzes/vivarium_research_alzheimers/results_tables


# Record version of Vivarium Helpers

In [3]:
!conda list | grep vivarium

# packages in environment at /ihme/homes/lutzes/.conda/envs/vivarium_csu_alzheimers_artifact:
vivarium-artifact         1.1.1                    pypi_0    pypi
vivarium-build-utils      4.11.1                   pypi_0    pypi
vivarium-cluster-tools    4.8.0                    pypi_0    pypi
vivarium-config-tree      5.2.1                    pypi_0    pypi
vivarium-csu-alzheimers   0.1.dev76+g982fe3fb1          pypi_0    pypi
vivarium-dependencies     1.3.1                    pypi_0    pypi
vivarium-engine           6.0.1                    pypi_0    pypi
vivarium-gbd-access       7.0.1                    pypi_0    pypi
vivarium-gbd-mapping      6.0.7                    pypi_0    pypi
vivarium-helpers          0.5.0.dev251111          pypi_0    pypi
vivarium-inputs           9.1.3                    pypi_0    pypi
vivarium-public-health    6.8.0                    pypi_0    pypi
vivarium-risk-distributions 3.2.1                    pypi_0    pypi


In [4]:
!pip freeze | grep vivarium

vivarium-artifact==1.1.1
vivarium-build-utils==4.11.1
vivarium-cluster-tools==4.8.0
vivarium-config-tree==5.2.1
-e git+https://github.com/ihmeuw/vivarium_csu_alzheimers.git@982fe3fb1cfe03824fcf5cf945850e1c9b0688b8#egg=vivarium_csu_alzheimers
vivarium-dependencies==1.3.1
vivarium-engine==6.0.1
vivarium-gbd-mapping==6.0.7
vivarium-public-health==6.8.0
vivarium-risk-distributions==3.2.1
vivarium_gbd_access==7.0.1
-e git+https://github.com/ihmeuw/vivarium_helpers.git@5b35fe7c0e45dd7296841aa977c3d299d1fdcaac#egg=vivarium_helpers
vivarium_inputs==9.1.3


# Define data directories

In [5]:
# Project directory
project_dir = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/')

# Output directory to store final results, in a folder named with
# today's date. These files include draws. write_final_csvs.ipynb writes
# the client files without draws to the no_draws folder next to it.
results_date = datetime.date.today().strftime('%Y_%m_%d')
output_dir = project_dir / 'results' / 'final_csvs' / results_date / 'draws'
output_dir.mkdir(parents=True, exist_ok=True)


def save_csv(df, filename):
    """Write df to output_dir, with a warning if the file already exists."""
    path = output_dir / filename
    if path.exists():
        warnings.warn(f"Overwriting existing file {path}")
    df.to_csv(path, index=False)


# Output from multistate life table
# mslt_output_dir = output_dir.parent / 'results_11_14_2025_mslt'
mslt_output_dir = project_dir / 'results' / 'mslt_draws'
print(mslt_output_dir.exists())
output_dir

True


PosixPath('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/final_csvs/2026_11_19/draws')

In [6]:
# # For testing: Run directory containing model 8.3 results for all
# # locations
# model_run_subdir = 'results/abie_consistent_model_test/united_states_of_america/2025_10_28_08_55_05/'

# # Results directory for model 8.3, for testing
# results_dirs = project_dir / model_run_subdir / 'results/'

# Artifact for models 10.0 - 12.2 was '10.0'
# Artifact for models 13.0 and 14.0
artifact_model_number = '13.0'


In [7]:
# Model 8.4 results (final runs for 10/31/25 intermediate results)
# Each batch run contains all locations, 100 random seeds, and 3 or 4 draws
model_run_dir_8_4 = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model8.4/model_spec')

# Model 8.7 results (updated final run completed on 11/10/2025)
model_run_dir_8_7 = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model8.7/model_spec')

# Model 12.2 results (updated final run completed on 02/14/2026)
model_run_dir_12_2 = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model12.2')

# Model 13.0 results (rebuilt artifacts, run started 10/29/2026)
# This run has a single run directory. The loader adds the 'results'
# subfolder to each run directory.
# The model 13.0 run is V&V sized (5 seeds of 20,000 simulants) and has
# only 3 locations, so use the V&V run type for the model scale.
run_type_13_0 = RunType.V_AND_V
locations_13_0 = ['United States of America', 'China', 'Brazil']
model_run_dir_13_0 = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model13.0/model_spec/2026_10_29_14_47_34')

# Model 14.0 results (new testing ramps and scenarios, run started 10/30/2026)
# Same artifacts as model 13.0. This run is also V&V sized (5 seeds of
# 20,000 simulants), with all 10 locations.
run_type_14_0 = RunType.V_AND_V
locations_14_0 = loading.LOCATIONS
model_run_dir_14_0 = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model14.0/model_spec/2026_10_30_13_30_29')

# Model 16.0 results (production run started 11/18/2026)
# Same artifacts as model 13.0 and the same scenarios as model 14.0.
# This run is production sized (100 seeds of 20,000 simulants), with all
# 10 locations and 25 draws.
run_type_16_0 = RunType.FINAL
locations_16_0 = loading.LOCATIONS
model_run_dir_16_0 = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model16.0/model_spec/2026_11_18_10_15_08')

# Batch runs are stored in "timestamp" subdirectories of the model run
# directory.
# In model 12.2, the run directories are nested under two more levels of
# subfolders: batch number, then model_spec, then timestamp.

# Select a model and filter for testing
# batch_run_dirs = loading.list_batch_run_directories(model_run_dir_12_2)#[:2]
# batch_run_dirs = [str(model_run_dir_13_0)]
# batch_run_dirs = [str(model_run_dir_14_0)]
batch_run_dirs = [str(model_run_dir_16_0)]

batch_run_dirs

['/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model16.0/model_spec/2026_11_18_10_15_08']

# Create an AlzheimersResultsProcessor object to process results

In [8]:
results = AlzheimersResultsProcessor(
    artifact_model_number,
    run_type_16_0,
    locations=locations_16_0,
    batch_run_dirs=batch_run_dirs,
)

In [9]:
results.initial_simulation_population

2000000

In [10]:
results.locations

['United States of America',
 'Brazil',
 'China',
 'Germany',
 'Israel',
 'Japan',
 'Spain',
 'Sweden',
 'Taiwan',
 'United Kingdom']

In [11]:
results.draws

[202,
 170,
 164,
 135,
 243,
 181,
 197,
 35,
 11,
 75,
 41,
 26,
 216,
 105,
 15,
 198,
 233,
 179,
 19,
 156,
 172,
 90,
 96,
 177,
 24]

# Load population data and calculate model scale

In [12]:
with Timer():
    results.load_population_data(append_aggregates=False)
results.ops.describe(results.model_scale)

Elapsed time: 0:00:12.758008


,count,mean,std,min,25%,50%,75%,max
location,,,,,,,,
Brazil,25.0,0.666241,0.048702,0.569066,0.647484,0.664228,0.697841,0.746260
China,25.0,0.074932,0.004991,0.066343,0.071189,0.073947,0.080082,0.082505
Germany,25.0,0.564469,0.046950,0.483817,0.541823,0.565487,0.574408,0.692730
Israel,25.0,26.437376,2.594453,21.371444,25.341007,26.388861,27.918803,33.464651
Japan,25.0,0.369508,0.022974,0.323476,0.355148,0.364880,0.390183,0.409644
Spain,25.0,1.898532,0.096439,1.775293,1.819973,1.873009,1.940022,2.072102
Sweden,25.0,7.868337,0.443373,7.053982,7.624894,7.863023,8.151224,8.754205
Taiwan,25.0,5.241038,0.277182,4.550281,5.137899,5.242813,5.410650,5.660475
United Kingdom,25.0,1.270413,0.060722,1.121093,1.232772,1.271746,1.314122,1.389850


In [13]:
results.person_time

,location,sex,event_year,input_draw,value,age_group
0,United States of America,Female,2025,202,1.128934e+07,25_to_29
1,United States of America,Female,2025,170,1.033839e+07,25_to_29
2,United States of America,Female,2025,164,1.120331e+07,25_to_29
3,United States of America,Female,2025,135,1.001506e+07,25_to_29
4,United States of America,Female,2025,243,1.131052e+07,25_to_29
...,...,...,...,...,...,...
569995,United Kingdom,Male,2100,172,1.325974e+05,95_plus
569996,United Kingdom,Male,2100,90,1.267402e+05,95_plus
569997,United Kingdom,Male,2100,96,1.354910e+05,95_plus
569998,United Kingdom,Male,2100,177,1.356587e+05,95_plus


In [14]:
570000*2

1140000

In [15]:
results.person_time.input_draw.unique()

array([202, 170, 164, 135, 243, 181, 197,  35,  11,  75,  41,  26, 216,
       105,  15, 198, 233, 179,  19, 156, 172,  90,  96, 177,  24],
      dtype=int16)

In [16]:
results.person_time.age_group.unique()

['25_to_29', '30_to_34', '35_to_39', '40_to_44', '45_to_49', ..., '75_to_79', '80_to_84', '85_to_89', '90_to_94', '95_plus']
Length: 15
Categories (15, object): ['25_to_29' < '30_to_34' < '35_to_39' < '40_to_44' ... '80_to_84' < '85_to_89' < '90_to_94' < '95_plus']

# Load prevalence and append counterfactual person-time

In [17]:
with Timer():
    results.load_prevalence_and_append_counterfactual_person_time(append_aggregates=True)
results.person_time

Timer BatchLoadingTimer started
[('age_group', '!=', '25_to_29'), ('event_year', '>=', '2025')]
{'all': PosixPath('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model16.0/model_spec/2026_11_18_10_15_08/results')}


36867.612507 MB after read_parquet


234.618534 MB after aggregating seeds and converting dtypes
234.618534 MB total
234.618534 MB after enforcing dtypes
Elapsed time: 937.8924 seconds


223.448606 MB measure


268.136474 MB minuend
44.696474 MB subtrahend


191.529254 MB minuend re-indexed
31.929254 MB subtrahend re-indexed


... (2 outputs left out)

/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


Elapsed time: 0:16:00.271196


,location,sex,event_year,input_draw,value,age_group,scenario
0,United States of America,Female,2025,202,1.063863e+07,65_to_69,baseline
1,United States of America,Female,2025,170,9.662163e+06,65_to_69,baseline
2,United States of America,Female,2025,164,1.057504e+07,65_to_69,baseline
3,United States of America,Female,2025,135,9.369922e+06,65_to_69,baseline
4,United States of America,Female,2025,243,1.067762e+07,65_to_69,baseline
...,...,...,...,...,...,...,...
6041995,United States of America,Both,2100,243,8.062193e+07,65_plus,baseline
6041996,United States of America,Both,2100,243,8.071912e+07,65_plus,intermediate_testing_with_treatment
6041997,United States of America,Both,2100,243,8.062193e+07,65_plus,intermediate_testing_zero_impact_treatment
6041998,United States of America,Both,2100,243,8.071912e+07,65_plus,slow_testing_with_treatment


In [18]:
results.person_time.age_group.unique()

['65_to_69', '70_to_74', '75_to_79', '80_to_84', '85_to_89', ..., '45_to_49', '50_to_54', '55_to_59', '60_to_64', '65_plus']
Length: 16
Categories (16, object): ['25_to_29' < '30_to_34' < '35_to_39' < '40_to_44' ... '85_to_89' < '90_to_94' < '95_plus' < '65_plus']

In [19]:
# Check that population is the same in baseline and bbbm_testing
# scenarios, and that it's greater in the treatment scenario
# NOTE: The baseline scenario still includes the 25-29 age group, so the
# count is higher than the other scenarios
results.person_time.groupby('scenario').value.describe()

,count,mean,std,min,25%,50%,75%,max
scenario,,,,,,,,
accelerated_testing_with_treatment,855000.0,9.880078e+06,2.594357e+07,5843.078093,710127.604065,2.443542e+06,7.443868e+06,4.123229e+08
accelerated_testing_zero_impact_treatment,855000.0,9.876404e+06,2.592948e+07,5843.078093,709768.395035,2.443440e+06,7.440779e+06,4.118594e+08
baseline,912000.0,9.754638e+06,2.532497e+07,5843.078093,722476.817775,2.453489e+06,7.525820e+06,4.118594e+08
intermediate_testing_with_treatment,855000.0,9.879990e+06,2.594325e+07,5843.078093,710083.791360,2.443542e+06,7.443868e+06,4.123229e+08
intermediate_testing_zero_impact_treatment,855000.0,9.876404e+06,2.592948e+07,5843.078093,709768.395035,2.443440e+06,7.440779e+06,4.118594e+08
slow_testing_with_treatment,855000.0,9.879885e+06,2.594285e+07,5843.078093,710081.941170,2.443542e+06,7.443842e+06,4.123229e+08
slow_testing_zero_impact_treatment,855000.0,9.876404e+06,2.592948e+07,5843.078093,709768.395035,2.443440e+06,7.440779e+06,4.118594e+08


# Process population data for output

In [20]:
person_time_processed = results.process_person_time()
person_time_processed

,location,sex,event_year,input_draw,value,age_group,scenario,measure,metric
0,United States of America,Female,2025,202,1.063863e+07,65_to_69,baseline,Population,Number
1,United States of America,Female,2025,170,9.662163e+06,65_to_69,baseline,Population,Number
2,United States of America,Female,2025,164,1.057504e+07,65_to_69,baseline,Population,Number
3,United States of America,Female,2025,135,9.369922e+06,65_to_69,baseline,Population,Number
4,United States of America,Female,2025,243,1.067762e+07,65_to_69,baseline,Population,Number
...,...,...,...,...,...,...,...,...,...
6041995,United States of America,Both,2100,243,8.062193e+07,65_plus,baseline,Population,Number
6041996,United States of America,Both,2100,243,8.071912e+07,65_plus,intermediate_testing_with_treatment,Population,Number
6041997,United States of America,Both,2100,243,8.062193e+07,65_plus,intermediate_testing_zero_impact_treatment,Population,Number
6041998,United States of America,Both,2100,243,8.071912e+07,65_plus,slow_testing_with_treatment,Population,Number


In [21]:
person_time_processed.age_group.unique()

['65_to_69', '70_to_74', '75_to_79', '80_to_84', '85_to_89', ..., '45_to_49', '50_to_54', '55_to_59', '60_to_64', '65_plus']
Length: 15
Categories (16, object): ['25_to_29' < '30_to_34' < '35_to_39' < '40_to_44' ... '85_to_89' < '90_to_94' < '95_plus' < '65_plus']

In [22]:
# We don't need to scale up or append rates and aggregates to the
# population data, so we just summarize and beautify
person_time_output = (
    person_time_processed
    .pipe(results.summarize, include_draws=True)
    .pipe(results.beautify)
)
person_time_output

Timer SummarizingTimer started


Elapsed time: 108.8070 seconds


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/projects/alzheimers/results.py:990: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  .replace(column_value_map)


,Year,Location,Age,Sex,Scenario,Measure,Metric,Mean,95% UI Lower,95% UI Upper,...,draw_172,draw_177,draw_179,draw_181,draw_197,draw_198,draw_202,draw_216,draw_233,draw_243
0,2025,Brazil,30_to_34,Female,"Accelerated Testing, With Treatment",Population,Number,8.758306e+06,7.912228e+06,1.030540e+07,...,9.176873e+06,7.994101e+06,9.402667e+06,1.074325e+07,8.974417e+06,8.968654e+06,1.001351e+07,8.349765e+06,8.584587e+06,7.979310e+06
1,2025,Brazil,30_to_34,Male,"Accelerated Testing, With Treatment",Population,Number,8.526407e+06,7.701830e+06,1.003098e+07,...,8.935294e+06,7.785616e+06,9.154593e+06,1.045730e+07,8.736962e+06,8.734136e+06,9.746771e+06,8.127765e+06,8.357095e+06,7.767043e+06
2,2025,Brazil,30_to_34,Both,"Accelerated Testing, With Treatment",Population,Number,1.728471e+07,1.561406e+07,2.033639e+07,...,1.811217e+07,1.577972e+07,1.855726e+07,2.120055e+07,1.771138e+07,1.770279e+07,1.976028e+07,1.647753e+07,1.694168e+07,1.574635e+07
3,2025,Brazil,30_to_34,Female,"Accelerated Testing, No Treatment",Population,Number,8.758306e+06,7.912228e+06,1.030540e+07,...,9.176873e+06,7.994101e+06,9.402667e+06,1.074325e+07,8.974417e+06,8.968654e+06,1.001351e+07,8.349765e+06,8.584587e+06,7.979310e+06
4,2025,Brazil,30_to_34,Male,"Accelerated Testing, No Treatment",Population,Number,8.526407e+06,7.701830e+06,1.003098e+07,...,8.935294e+06,7.785616e+06,9.154593e+06,1.045730e+07,8.736962e+06,8.734136e+06,9.746771e+06,8.127765e+06,8.357095e+06,7.767043e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239395,2100,United States of America,65_plus,Male,"Slow Testing, With Treatment",Population,Number,3.519404e+07,3.240497e+07,3.692847e+07,...,3.372532e+07,3.563478e+07,3.695843e+07,3.527597e+07,3.679891e+07,3.492584e+07,3.555130e+07,3.474214e+07,3.422612e+07,3.661803e+07
239396,2100,United States of America,65_plus,Both,"Slow Testing, With Treatment",Population,Number,7.784175e+07,7.158418e+07,8.160809e+07,...,7.460517e+07,7.881497e+07,8.200904e+07,7.846566e+07,8.134080e+07,7.753858e+07,7.891927e+07,7.678954e+07,7.590779e+07,8.071912e+07
239397,2100,United States of America,65_plus,Female,"Slow Testing, No Treatment",Population,Number,4.258746e+07,3.911701e+07,4.467700e+07,...,4.083059e+07,4.313751e+07,4.498118e+07,4.312872e+07,4.447421e+07,4.255366e+07,4.332381e+07,4.199935e+07,4.162717e+07,4.403265e+07
239398,2100,United States of America,65_plus,Male,"Slow Testing, No Treatment",Population,Number,3.516898e+07,3.237755e+07,3.690257e+07,...,3.370528e+07,3.561789e+07,3.692990e+07,3.525218e+07,3.677136e+07,3.490192e+07,3.553116e+07,3.472089e+07,3.420262e+07,3.658928e+07


In [23]:
# Save population data
save_csv(person_time_output, 'population.csv')

# Deaths and averted deaths

In [24]:
with Timer():
    results.load_measure_from_batch_runs('deaths', filters=loading.FINAL_RESULTS_FILTERS['deaths'])

print(len(results.data.deaths), 'rows')
results.data.deaths.head()

Timer BatchLoadingTimer started
[('age_group', '!=', '25_to_29'), ('event_year', '>=', '2025')]
{'all': PosixPath('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model16.0/model_spec/2026_11_18_10_15_08/results')}


24578.412428 MB after read_parquet


148.966201 MB after aggregating seeds and converting dtypes
148.966201 MB total
148.966201 MB after enforcing dtypes
Elapsed time: 929.0873 seconds
Elapsed time: 0:15:29.087484
7448000 rows


,age_group,artifact_path,entity,entity_type,event_year,input_draw,measure,scenario,sex,value,location
0,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_state,cause,2025,11,deaths,baseline,Female,0.0,Japan
1,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_state,cause,2025,11,deaths,baseline,Male,0.0,Japan
2,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_state,cause,2025,11,deaths,slow_testing_zero_impact_treatment,Female,0.0,Japan
3,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_state,cause,2025,11,deaths,slow_testing_zero_impact_treatment,Male,0.0,Japan
4,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_state,cause,2025,11,deaths,intermediate_testing_zero_impact_treatment,Female,0.0,Japan


In [25]:
# Check that draws in the data match expected draws from the keyspace metadata
assert set(results.data.deaths.input_draw.unique()) == set(results.draws)

In [26]:
deaths_prepped = results.process_deaths(results.data.deaths)
deaths_prepped

148.966201 MB measure


29.798069 MB minuend
178.758069 MB subtrahend


19.16053 MB minuend re-indexed
114.92053 MB subtrahend re-indexed


114.920594 MB difference
127.686201 MB difference with reset index
134.071121 MB final difference


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/projects/alzheimers/results.py:334: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  disease_stage=lambda df: df['entity'].replace(


,age_group,artifact_path,entity,entity_type,event_year,input_draw,measure,scenario,sex,value,location,metric,disease_stage
0,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_state,cause,2025,11,Deaths Associated with AD,baseline,Female,0.0,Japan,Number,alzheimers_disease_state
1,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_state,cause,2025,11,Deaths Associated with AD,baseline,Male,0.0,Japan,Number,alzheimers_disease_state
2,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_state,cause,2025,11,Deaths Associated with AD,slow_testing_zero_impact_treatment,Female,0.0,Japan,Number,alzheimers_disease_state
3,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_state,cause,2025,11,Deaths Associated with AD,slow_testing_zero_impact_treatment,Male,0.0,Japan,Number,alzheimers_disease_state
4,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_state,cause,2025,11,Deaths Associated with AD,intermediate_testing_zero_impact_treatment,Female,0.0,Japan,Number,alzheimers_disease_state
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13831995,95_plus,/mnt/team/simulation_science/pub/models/vivari...,other_causes,cause,2100,243,Averted Deaths Due to Other Causes,slow_testing_with_treatment,Male,-402.0,Israel,Number,all
13831996,95_plus,/mnt/team/simulation_science/pub/models/vivari...,other_causes,cause,2100,243,Averted Deaths Due to Other Causes,intermediate_testing_with_treatment,Female,-881.0,Israel,Number,all
13831997,95_plus,/mnt/team/simulation_science/pub/models/vivari...,other_causes,cause,2100,243,Averted Deaths Due to Other Causes,intermediate_testing_with_treatment,Male,-402.0,Israel,Number,all
13831998,95_plus,/mnt/team/simulation_science/pub/models/vivari...,other_causes,cause,2100,243,Averted Deaths Due to Other Causes,accelerated_testing_with_treatment,Female,-881.0,Israel,Number,all


In [27]:
deaths_prepped.query("0 < abs(value) < 1").value.describe()

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: value, dtype: float64

In [28]:
# deaths_output = results.summarize_and_beautify(deaths_prepped, 'entity')
# process_* returns event_year as an unordered categorical, and the
# event_year >= 2025 filter in do_final_processing cannot compare it
# with the installed pandas version, so cast it back to int16 first
deaths_output = results.do_final_processing(deaths_prepped.astype({'event_year': 'int16'}), include_draws=True)
deaths_output

2026-11-19 18:05:37.347337


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


Timer SummarizingTimer started


Elapsed time: 837.5570 seconds


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/projects/alzheimers/results.py:990: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  .replace(column_value_map)


2026-11-19 18:20:51.943895


,Year,Location,Age,Sex,Disease Stage,Scenario,Measure,Metric,Mean,95% UI Lower,...,draw_172,draw_177,draw_179,draw_181,draw_197,draw_198,draw_202,draw_216,draw_233,draw_243
0,2025,Japan,30_to_34,Female,AD Dementia,"Slow Testing, No Treatment",Averted Deaths Associated with AD,Number,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2025,Japan,30_to_34,Male,AD Dementia,"Slow Testing, No Treatment",Averted Deaths Associated with AD,Number,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2025,Japan,30_to_34,Both,AD Dementia,"Slow Testing, No Treatment",Averted Deaths Associated with AD,Number,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,2025,Japan,30_to_34,Female,AD Dementia,"Intermediate Testing, No Treatment",Averted Deaths Associated with AD,Number,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,2025,Japan,30_to_34,Male,AD Dementia,"Intermediate Testing, No Treatment",Averted Deaths Associated with AD,Number,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1778395,2100,Israel,65_plus,Male,All,"Intermediate Testing, With Treatment",Deaths Due to Other Causes,Rate,0.004572,0.004145,...,0.004399,0.004373,0.004759,0.004671,0.004363,0.004964,0.004720,0.004482,0.005084,0.004181
1778396,2100,Israel,65_plus,Both,All,"Intermediate Testing, With Treatment",Deaths Due to Other Causes,Rate,0.005564,0.005068,...,0.005781,0.005333,0.005715,0.005813,0.005225,0.005729,0.005769,0.005521,0.005614,0.005167
1778397,2100,Israel,65_plus,Female,All,"Accelerated Testing, With Treatment",Deaths Due to Other Causes,Rate,0.006446,0.005811,...,0.007008,0.006178,0.006565,0.006829,0.005990,0.006412,0.006703,0.006434,0.006083,0.006046
1778398,2100,Israel,65_plus,Male,All,"Accelerated Testing, With Treatment",Deaths Due to Other Causes,Rate,0.004572,0.004145,...,0.004399,0.004373,0.004759,0.004671,0.004363,0.004964,0.004720,0.004482,0.005084,0.004181


In [29]:
save_csv(deaths_output, "deaths.csv")

# Prevalence

In [30]:
filename = 'person_time_alzheimers_disease_and_other_dementias'
prevalence_short_name = 'alzheimers_person_time'

# We should have loaded this when processing population above, but just
# in case we didn't, load it here
if prevalence_short_name not in results.data:
    with Timer():
        results.load_measure_from_batch_runs(filename, prevalence_short_name, filters=loading.FINAL_RESULTS_FILTERS[filename])

print(len(results.data[prevalence_short_name]), 'rows')
results.data[prevalence_short_name].head()

11172000 rows


,age_group,artifact_path,entity,entity_type,event_year,input_draw,measure,scenario,sex,sub_entity,value,location
0,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2025,11,person_time,baseline,Female,alzheimers_blood_based_biomarker_state,0.0,Japan
1,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2025,11,person_time,baseline,Female,alzheimers_mild_cognitive_impairment_state,0.0,Japan
2,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2025,11,person_time,baseline,Female,alzheimers_disease_state,0.0,Japan
3,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2025,11,person_time,baseline,Male,alzheimers_blood_based_biomarker_state,0.0,Japan
4,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2025,11,person_time,baseline,Male,alzheimers_mild_cognitive_impairment_state,0.0,Japan


In [31]:
results.data.alzheimers_person_time.query("age_group == '30_to_34' and value > 0")

,age_group,artifact_path,entity,entity_type,event_year,input_draw,measure,scenario,sex,sub_entity,value,location


In [32]:
processed_prevalence_short_name = f'{prevalence_short_name}_processed'

# We should have saved processed prevalence data when processing
# population above, but just in case we didn't, process it here
if processed_prevalence_short_name not in results.data:
    results.data[processed_prevalence_short_name] = results.process_prevalence_counts(results.data[prevalence_short_name])

prevalence_counts = results.data[processed_prevalence_short_name]
print_memory_usage(prevalence_counts)
print(len(prevalence_counts), 'rows')
prevalence_counts.tail()

414.968698 MB 
20748000 rows


,age_group,artifact_path,entity,entity_type,event_year,input_draw,location,measure,scenario,sex,disease_stage,value,metric
20747995,95_plus,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2100,243,Israel,Change in Prevalence from Reference,accelerated_testing_with_treatment,Female,alzheimers_mild_cognitive_impairment_state,1397.700205,Number
20747996,95_plus,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2100,243,Israel,Change in Prevalence from Reference,accelerated_testing_with_treatment,Female,alzheimers_disease_state,1288.076660,Number
20747997,95_plus,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2100,243,Israel,Change in Prevalence from Reference,accelerated_testing_with_treatment,Male,alzheimers_blood_based_biomarker_state,545.127995,Number
20747998,95_plus,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2100,243,Israel,Change in Prevalence from Reference,accelerated_testing_with_treatment,Male,alzheimers_mild_cognitive_impairment_state,836.626968,Number
20747999,95_plus,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2100,243,Israel,Change in Prevalence from Reference,accelerated_testing_with_treatment,Male,alzheimers_disease_state,829.152635,Number


In [33]:
prevalence_counts.query("value < 0")[['scenario', 'measure', 'disease_stage']].value_counts()

scenario                             measure                              disease_stage                             
accelerated_testing_with_treatment   Change in Prevalence from Reference  alzheimers_disease_state                      208277
intermediate_testing_with_treatment  Change in Prevalence from Reference  alzheimers_disease_state                      208248
slow_testing_with_treatment          Change in Prevalence from Reference  alzheimers_disease_state                      208243
                                                                          alzheimers_mild_cognitive_impairment_state    161338
intermediate_testing_with_treatment  Change in Prevalence from Reference  alzheimers_mild_cognitive_impairment_state    159983
accelerated_testing_with_treatment   Change in Prevalence from Reference  alzheimers_mild_cognitive_impairment_state    158938
Name: count, dtype: int64

In [34]:
# Sometimes we increase the prevalence of MCI or dementia in some age groups
prevalence_counts.query(
    "measure == 'Change in Prevalence from Reference' and value > 0"
    " and disease_stage != 'alzheimers_blood_based_biomarker_state' "
    ).age_group.value_counts()

age_group
95_plus     169250
90_to_94     92593
85_to_89     55074
40_to_44         0
25_to_29         0
30_to_34         0
35_to_39         0
55_to_59         0
50_to_54         0
45_to_49         0
60_to_64         0
75_to_79         0
70_to_74         0
65_to_69         0
80_to_84         0
Name: count, dtype: int64

In [35]:
prevalence_counts.query(
    "measure == 'Change in Prevalence from Reference' and value > 0"
    " and disease_stage != 'alzheimers_blood_based_biomarker_state'"
    " and age_group in ['65_to_69', '80_to_84']"
    )

,age_group,artifact_path,entity,entity_type,event_year,input_draw,location,measure,scenario,sex,disease_stage,value,metric


In [36]:
prevalence_counts.query("0 < abs(value) < 1")

,age_group,artifact_path,entity,entity_type,event_year,input_draw,location,measure,scenario,sex,disease_stage,value,metric
1603817,40_to_44,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2032,135,Japan,Prevalence,baseline,Male,alzheimers_disease_state,0.498289,Number
1603823,40_to_44,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2032,135,Japan,Prevalence,slow_testing_zero_impact_treatment,Male,alzheimers_disease_state,0.498289,Number
1603829,40_to_44,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2032,135,Japan,Prevalence,intermediate_testing_zero_impact_treatment,Male,alzheimers_disease_state,0.498289,Number
1603835,40_to_44,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2032,135,Japan,Prevalence,accelerated_testing_zero_impact_treatment,Male,alzheimers_disease_state,0.498289,Number
1603841,40_to_44,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2032,135,Japan,Prevalence,slow_testing_with_treatment,Male,alzheimers_disease_state,0.498289,Number
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20700395,95_plus,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2048,19,Israel,Change in Prevalence from Reference,slow_testing_with_treatment,Male,alzheimers_disease_state,0.996578,Number
20700467,95_plus,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2048,26,Israel,Change in Prevalence from Reference,slow_testing_with_treatment,Male,alzheimers_disease_state,-0.996578,Number
20700647,95_plus,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2048,96,Israel,Change in Prevalence from Reference,slow_testing_with_treatment,Male,alzheimers_disease_state,0.996578,Number
20700761,95_plus,/mnt/team/simulation_science/pub/models/vivari...,alzheimers_disease_and_other_dementias,cause,2048,156,Israel,Change in Prevalence from Reference,intermediate_testing_with_treatment,Male,alzheimers_disease_state,0.996578,Number


In [37]:
prevalence_counts.query("abs(value) >= 1e-9")['value'].describe() # count 4.940696e+06

count    1.171271e+07
mean     4.966447e+04
std      7.875051e+04
min     -1.860660e+04
25%      1.169982e+03
50%      1.263935e+04
75%      6.717332e+04
max      7.041210e+05
Name: value, dtype: float64

In [38]:
results.data.keys()

dict_keys(['ops', 'alzheimers_person_time', 'alzheimers_person_time_processed', 'deaths'])

In [39]:
with Timer():
    # prevalence_output = results.summarize_and_beautify(prevalence_counts)
    # process_* returns event_year as an unordered categorical, and the
    # event_year >= 2025 filter in do_final_processing cannot compare it
    # with the installed pandas version, so cast it back to int16 first
    prevalence_output = results.do_final_processing(prevalence_counts.astype({'event_year': 'int16'}), include_draws=True)
print_memory_usage(prevalence_output)
print(len(prevalence_output), 'rows')
prevalence_output.tail()

2026-11-19 18:21:27.286767


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


Timer SummarizingTimer started


Elapsed time: 1249.1439 seconds


2026-11-19 18:43:58.085328
Elapsed time: 0:22:30.998001
618.889377 MB 
2667600 rows


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/projects/alzheimers/results.py:990: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  .replace(column_value_map)


,Year,Location,Age,Sex,Disease Stage,Scenario,Measure,Metric,Mean,95% UI Lower,...,draw_172,draw_177,draw_179,draw_181,draw_197,draw_198,draw_202,draw_216,draw_233,draw_243
2667595,2100,Israel,65_plus,Male,AD Dementia,"Intermediate Testing, With Treatment",Prevalence,Rate,0.022354,0.020248,...,0.021228,0.021860,0.022693,0.022398,0.021422,0.023437,0.023602,0.021546,0.023703,0.021372
2667596,2100,Israel,65_plus,Both,AD Dementia,"Intermediate Testing, With Treatment",Prevalence,Rate,0.029938,0.027513,...,0.030877,0.029590,0.029999,0.030213,0.028314,0.030181,0.031562,0.030650,0.030446,0.028396
2667597,2100,Israel,65_plus,Female,AD Dementia,"Accelerated Testing, With Treatment",Prevalence,Rate,0.036675,0.033496,...,0.039446,0.036405,0.036496,0.037168,0.034425,0.036205,0.038646,0.038653,0.036407,0.034649
2667598,2100,Israel,65_plus,Male,AD Dementia,"Accelerated Testing, With Treatment",Prevalence,Rate,0.022354,0.020248,...,0.021228,0.021860,0.022693,0.022398,0.021422,0.023437,0.023602,0.021546,0.023703,0.021372
2667599,2100,Israel,65_plus,Both,AD Dementia,"Accelerated Testing, With Treatment",Prevalence,Rate,0.029938,0.027513,...,0.030877,0.029590,0.029999,0.030213,0.028314,0.030181,0.031562,0.030650,0.030446,0.028396


In [40]:
# Save prevalence data
save_csv(prevalence_output, 'prevalence.csv')

In [41]:
!date

Thu Nov 19 18:44:50 PST 2026
